# Sentiment Analysis Test Run for a Single Theme Group
This sentiment analysis test run looked at the theme group 'environment'. While preforming the cleaning Tianrun realized that during preprocessing Camilla had accidently run the preprocessing on the description of the Ted Talks instead of the actual transcripts. Since then Camilla has completed the preprocessing on the actual transcripts we have to had the opportunity yet to merge the preprocessed transcript columns and the cleaned data.

Therefore, in this short test run the sentiment analysis is done on the description. Although not optimal, this is also a check to see how the three different methods : roberta, NRC Lexicon and VADER work

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers scipy


In [ ]:
!pip install -q nrclex==3.0.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.4/396.4 kB 8.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
!pip install vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.6 MB/s eta 0:00:00


In [ ]:

import pandas as pd


# Check the difference bettween title and name
ted_master = pd.read_csv('/content/drive/MyDrive/School/AUC/Semester 4/Text Mining/ted_master.csv')

print("Printing the theme group counts")
print(ted_master['theme'].value_counts())

ted_master_filtered = ted_master[ted_master['theme'] != 'unclassified']

print(" Printing theme groups after filtering out 'unclassified' ")
print(ted_master_filtered['theme'].value_counts())

ted_themes = ted_master_filtered[['title','theme','clean_text']]


environment = ted_master[ted_master['theme'] == 'environment']

# See how many and which talks are unclassified
print("Filtering only the environment ")
environment[['title', 'theme', 'clean_text']].head(10)

# Print the head to check the shape
small_sample = environment.head(10)

# Creating a text variable to store 'clean_text'
texts = small_sample['clean_text'].tolist()


Printing the theme group counts
theme
unclassified           481
environment            401
business               385
psychology             352
politics_governance    331
culture_society        252
education              251
Name: count, dtype: int64
 Printing theme groups after filtering out 'unclassified' 
theme
environment            401
business               385
psychology             352
politics_governance    331
culture_society        252
education              251
Name: count, dtype: int64
Filtering only the environment 


In [ ]:
from transformers import pipeline

# Initialize the sentiment analysis pipeline
sentiment_task = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest")

# Run sentiment analysis
results = sentiment_task(texts)

# Print results
for text, res in zip(texts, results):
    print(f"Text: {text} | Label: {res['label']} | Score: {res['score']:.4f}")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Text: ['humor', 'humanity', 'exude', 'inconvenient', 'truth', 'al', 'gore', 'spell', '15', 'way', 'individual', 'address', 'climate', 'change', 'immediately', 'buy', 'hybrid', 'invent', 'new', 'hotter', 'brand', 'global', 'warming'] | Label: neutral | Score: 0.8674
Text: ['emotionally', 'charge', 'talk', 'macarthur', 'win', 'activist', 'majora', 'carter', 'detail', 'fight', 'environmental', 'justice', 'south', 'bronx', 'show', 'minority', 'neighborhood', 'suffer', 'flawed', 'urban', 'policy'] | Label: neutral | Score: 0.7809
Text: ['designer', 'ross', 'lovegrove', 'expound', 'philosophy', 'fat', 'free', 'design', 'offer', 'insight', 'extraordinary', 'product', 'include', 'ty', 'nant', 'water', 'bottle', 'chair'] | Label: neutral | Score: 0.7837
Text: ['legendary', 'scientist', 'david', 'deutsch', 'put', 'theoretical', 'physics', 'burner', 'discuss', 'urgent', 'matter', 'survival', 'specie', 'step', 'solve', 'global', 'warming', 'say', 'admit', 'problem'] | Label: neutral | Score: 0.832

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Set up sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

# Run sentiment analysis
for sentence in texts:
    vs = analyzer.polarity_scores(sentence)
    print("{:-<65} {}".format(sentence, str(vs)))

['humor', 'humanity', 'exude', 'inconvenient', 'truth', 'al', 'gore', 'spell', '15', 'way', 'individual', 'address', 'climate', 'change', 'immediately', 'buy', 'hybrid', 'invent', 'new', 'hotter', 'brand', 'global', 'warming'] {'neg': 0.088, 'neu': 0.693, 'pos': 0.219, 'compound': 0.3818}
['emotionally', 'charge', 'talk', 'macarthur', 'win', 'activist', 'majora', 'carter', 'detail', 'fight', 'environmental', 'justice', 'south', 'bronx', 'show', 'minority', 'neighborhood', 'suffer', 'flawed', 'urban', 'policy'] {'neg': 0.284, 'neu': 0.494, 'pos': 0.222, 'compound': -0.25}
['designer', 'ross', 'lovegrove', 'expound', 'philosophy', 'fat', 'free', 'design', 'offer', 'insight', 'extraordinary', 'product', 'include', 'ty', 'nant', 'water', 'bottle', 'chair'] {'neg': 0.0, 'neu': 0.837, 'pos': 0.163, 'compound': 0.5106}
['legendary', 'scientist', 'david', 'deutsch', 'put', 'theoretical', 'physics', 'burner', 'discuss', 'urgent', 'matter', 'survival', 'specie', 'step', 'solve', 'global', 'warmi

In [ ]:
from nrclex import NRCLex
import nltk
import ast
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

# Separate title and the texts
texts = small_sample['clean_text'].apply(ast.literal_eval).tolist()
titles = small_sample['title'].tolist()

# Run Sentiment Analysis
for title,lst in zip(titles,texts):

  # Make a string because NRC Lexicon works with strings
  string = " ".join(lst)

  emotion = NRCLex(string)
  sentiment_analysis = NRCLex(string)

  print(f"Printing sentiment analysis for: {title}")
  print(sentiment_analysis.affect_frequencies)
  print('---\n')


Printing sentiment analysis for: Averting the climate crisis
{'fear': 0.1111111111111111, 'anger': 0.1111111111111111, 'anticip': 0.0, 'trust': 0.1111111111111111, 'surprise': 0.0, 'positive': 0.16666666666666666, 'negative': 0.16666666666666666, 'sadness': 0.1111111111111111, 'disgust': 0.1111111111111111, 'joy': 0.05555555555555555, 'anticipation': 0.05555555555555555}
---

Printing sentiment analysis for: Greening the ghetto
{'fear': 0.09090909090909091, 'anger': 0.09090909090909091, 'anticip': 0.0, 'trust': 0.2727272727272727, 'surprise': 0.0, 'positive': 0.18181818181818182, 'negative': 0.2727272727272727, 'sadness': 0.0, 'disgust': 0.0, 'joy': 0.0, 'anticipation': 0.09090909090909091}
---

Printing sentiment analysis for: Organic design, inspired by nature
{'fear': 0.0, 'anger': 0.0, 'anticip': 0.0, 'trust': 0.0, 'surprise': 0.0, 'positive': 0.5714285714285714, 'negative': 0.14285714285714285, 'sadness': 0.14285714285714285, 'disgust': 0.14285714285714285, 'joy': 0.0}
---

Printi

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [18]:
!pip install nbconvert
!jupyter nbconvert --to notebook --inplace --ClearMetadataPreprocessor.enabled=True Test_Run_Sentiment_Analysis.ipynb

[NbConvertApp] WARNING | pattern 'Test_Run_Sentiment_Analysis.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=Tru